# 02 – Vorverarbeitung 

hier werden die Beschwerdetexte der Stichprobe in  Wortlisten umgewandelt, die später vektorisiert und  ausgewertet werden.

**Eingabe:** 
Stichprobe von 20.000 Beschwerden, gezogen aus den 366.945 Beschwerden mit Text ohne Duplikate

**Ausgabe:** 
pro Beschwerde eine Liste bereinigter Wörter 

**Schritte:**
1. Platzhalter wie „XXXX" und Datumsmuster mit regulären Ausdrücken entfernen
2. Texte mit spaCy (`en_core_web_sm`) in Wörter zerlegen und auf Grundformen bringen
3. Filtern: nur Nomen, Verben und Adjektive ohne Stoppwörter, mindestens drei Buchstaben
4. Auffälligkeiten prüfen und dokumentieren (z. B. fehlerhafte Grundformen)

In [ ]:
import re
import pandas as pd
import spacy
from collections import Counter
import time

In [ ]:
sample = pd.read_csv("data/sample_20000.csv")
col = "Consumer complaint narrative"
print(sample.shape)
sample.head(3)

In [ ]:
def clean_text(text):
    text = re.sub(r"X{2,}(/X{2,})*(/\d{2,4})?", " ", text)  # XXXX, XX/XX/2018
    text = re.sub(r"\{\$[\d.,]+\}", " ", text)  # Beträge wie {$100.00}
    text = re.sub(r"\s+", " ", text).strip()  # mehrfache Leerzeichen
    return text


beispiel = sample[col].iloc[0]
print(beispiel[:300])
print("---")
print(clean_text(beispiel)[:300])

In [ ]:
sample["text_clean"] = sample[col].apply(clean_text)

print("Texte mit XX vorher:", sample[col].str.contains("XX").sum())
print("Texte mit XX nachher:", sample["text_clean"].str.contains("XX").sum())

In [ ]:
nlp = spacy.load("en_core_web_sm")

doc = nlp(sample["text_clean"].iloc[0][:300])

for token in doc:
    print(
        f"{token.text:15} {token.lemma_:15} {token.pos_:8} Stoppwort: {token.is_stop}"
    )

In [ ]:
ALLOWED_POS = {"NOUN", "VERB", "ADJ"}


def extract_tokens(doc):
    return [
        token.lemma_.lower()
        for token in doc
        if token.is_alpha
        and not token.is_stop
        and token.pos_ in ALLOWED_POS
        and len(token.lemma_) > 2
    ]


print(extract_tokens(doc))

## Filterregeln 

**Ablauf:** 
Zuerst entferne ich Platzhalter mit Regex. Danach zerlegt spaCy die Texte in Wörter und bildet den Stamm. Erst am Ende schreibe ich  klein, weil spaCy Wortarten am Originaltext zuverlässiger erkennt.

**Behalten werden** 
Wörter, die
- nur aus Buchstaben bestehen,
- keine Stoppwörter sind,
- Nomen, Verben oder Adjektive sind,
- mehr als zwei Buchstaben haben.

**Begründung**
- **Eigennamen verworfen:** Eigennamen sind in der Regel irrelevant und erhöhen unnötiges Rauschen
- **Adverbien verworfen:** Wörter wie „currently" tragen wenig zum Thema bei.
- **Verben behalten:** Sie tragen zur Semantik bei
- **Mindestlänge über zwei Buchstaben:** So fallen Kürzel weg, aber kurze Wörter wie „car" oder „bus" bleiben erhalten.



In [ ]:
test = sample["text_clean"].head(1000)

docs = list(nlp.pipe(test, disable=["parser", "ner"], batch_size=100))
tokens_test = [extract_tokens(d) for d in docs]

print(len(tokens_test))
print(tokens_test[0])
print(tokens_test[1])

## Auffälligkeit: Lemmatisierungsfehler bei „taxes"

**Problem:**
Das Modell `en_core_web_sm` bildet aus „taxes" in 63 von 64 Fällen die Grundform „taxis" statt „tax". Dieser Fehler beruht auf der falschen Annahme, dass hier aus der Endung -es die Grundform mit -is gebildet werden muss (Vergleiche z.B.: "crises" -> "crisis")

**Vorgehen:**
Ich habe an 1.000 Texten geprüft, ob es weitere falsche Grundformen gibt, indem ich die 200 häufigsten Paare aus Originalwort und Grundform durchgelesen habe.

**Ergebnis der Prüfung:**
„taxes → taxis" ist der einzige aufgefallene Fehler. Alle anderen Paare sind korrekte Grundformen.

**Entscheidung:**
- Da es sich nur um diesen Einzelfall handelt, bleibe ich bei `en_core_web_sm` und korrigiere die Grundformen nachträglich mit der Tabelle `{"taxis": "tax", "taxes": "tax"}`. „taxes" ist enthalten, weil es in einem Fall nicht verändert wurde und sonst unkorrigiert bliebe.
- Wären mehrere häufige Fehler aufgetaucht, hätte ich zuerst `en_core_web_md` und danach gegebenenfalls `en_core_web_trf` getestet.

**Anmerkung:**
Die Korrektur ersetzt auch ein echtes „taxis" (Plural von „taxi"). In den 1000 Testtexten kommt „taxis" im Originaltext nicht vor, nur „taxi" im Singular (2-mal), das von der Korrektur nicht verändert wird. Das Risiko ist daher vernachlässigbar.

In [ ]:
# wie oft kommt "taxis" und "tax" vor
counter = Counter(word for tokens in tokens_test for word in tokens)
print("taxis:", counter["taxis"], "| tax:", counter["tax"])

In [ ]:
# wird "taxes" immer falsch lemmatisiert
taxes_lemmas = Counter(
    t.lemma_.lower() for d in docs for t in d if t.text.lower() == "taxes"
)
print(taxes_lemmas)

In [ ]:
pairs = Counter(
    (t.text.lower(), t.lemma_.lower())
    for d in docs
    for t in d
    if t.is_alpha and not t.is_stop and t.pos_ in ALLOWED_POS and len(t.lemma_) > 2
)

changed = [(w, l, c) for (w, l), c in pairs.items() if w != l]
changed.sort(key=lambda x: -x[2])

for w, l, c in changed[:60]:
    print(f"{w:20} -> {l:20} {c}")

In [ ]:
for w, l, c in changed[60:200]:
    print(f"{w:20} -> {l:20} {c}")

## Verarbeitung aller 20.000 Texte

Die festgelegten Filterregeln werden auf die gesamte Stichprobe angewendet. Zusätzlich greift eine Korrektur-Tabelle (`LEMMA_FIXES`) für fehlerhafte Grundformen und  Tippfehler. Stoppwörter werden auch auf der Grundform geprüft.

**Laufzeit:** 
252,9 s für 20.000 Texte (ca. 4 Minuten, 12,6 ms pro Text).

**Kontrolle nach dem Lauf:**
- Die acht fehlerhaften Formen kommen im Ergebnis nicht mehr vor.
- Die korrigierten Wörter treten in den erwarteten Häufigkeiten auf

**Bewusst nicht angepasst:**
- „report" und „reporting" bleiben ohne Sonderregel, um die Zahl der Einzelfallkorrekturen klein zu halten.
- Mehrdeutige Grundformen („state", „lie") bleiben bestehen und fallen kaum ins Gewicht.

In [ ]:
# Regeln
ALLOWED_POS = {"NOUN", "VERB", "ADJ"}
STOPWORDS = nlp.Defaults.stop_words
MIN_LENGTH = 3  # mehr als 2 Buchstaben, kein Stoppwort
LEMMA_FIXES = {
    "taxis": "tax",
    "taxes": "tax",
    "pende": "pending",
    "datum": "data",
    "monie": "money",
    "reinserte": "reinsert",
    "hop": "hope",
    "recieve": "receive",  # sehr häufiger Tippfehler
}


def extract_tokens(doc, pair_counts=None):
    tokens = []
    for token in doc:
        if not token.is_alpha or token.is_stop or token.pos_ not in ALLOWED_POS:
            continue

        raw_lemma = token.lemma_.lower()
        lemma = LEMMA_FIXES.get(raw_lemma, raw_lemma)

        if len(lemma) < MIN_LENGTH or lemma in STOPWORDS:
            continue

        if pair_counts is not None and token.text.lower() != raw_lemma:
            pair_counts[(token.text.lower(), raw_lemma)] += 1

        tokens.append(lemma)
    return tokens

In [ ]:
texts = sample["text_clean"]
pair_counts = Counter()  # bei jedem Lauf wieder bei 0 starten
tokens_all = []

start = time.perf_counter()
for doc in nlp.pipe(texts, disable=["parser", "ner"], batch_size=100):
    tokens_all.append(extract_tokens(doc, pair_counts))
dauer = time.perf_counter() - start

sample["tokens"] = tokens_all
print(
    f"{len(texts)} Texte in {dauer:.1f} s ({dauer / len(texts) * 1000:.1f} ms pro Text)"
)

In [ ]:
all_tokens = [tok for toks in sample["tokens"] for tok in toks]
token_counts = Counter(all_tokens)

print("Falsche Grundformen (sollten 0 sein):")
for wrong in LEMMA_FIXES:
    print(f"  {wrong:10} {token_counts[wrong]}")

print("Korrigierte Grundformen:")
for right in sorted(set(LEMMA_FIXES.values())):
    print(f"  {right:10} {token_counts[right]}")

print(
    "call:",
    token_counts["call"],
    "| report:",
    token_counts["report"],
    "| reporting:",
    token_counts["reporting"],
)

In [ ]:
#  häufigste Paare
for (w, l), c in pair_counts.most_common(300):
    print(f"{w:20} -> {l:20} {c}")

In [ ]:
#  häufigste Wörter
Counter(all_tokens).most_common(100)

In [ ]:
kandidaten = [
    "call",
    "say",
    "get",
    "take",
    "give",
    "tell",
    "receive",
    "send",
    "ask",
    "pay",
]
print("Stoppwörter:", sorted(w for w in kandidaten if w in STOPWORDS))
print("Keine Stoppwörter:", sorted(w for w in kandidaten if w not in STOPWORDS))

In [ ]:
# Wörter 300-600
for (w, l), c in pair_counts.most_common()[300:600]:
    print(f"{w:20} -> {l:20} {c}")

**Info** spaCy bildet für Wörter mit mehreren Bedeutungen dieselbe Grundform, zum Beispiel „state" (Verb „angeben" und US-Bundesstaat) und „lie" („lügen" und „liegen"). Die Fälle sind selten und ändern die Ergebnisse voraussichtlich kaum.

In [ ]:
#checke sehr seltene und sehr häufige Wörter
doc_freq = Counter(w for toks in sample["tokens"] for w in set(toks))
n_docs = len(sample)

for w, c in doc_freq.most_common(60):
    print(f"{w:15} {c:6} {c / n_docs:6.1%}")

In [ ]:
#Abfrage für Untergrenze
n_docs = len(sample)
print("Verschiedene Wörter insgesamt:", len(doc_freq))

for min_df in [2, 5, 10, 20, 50]:
    n = sum(1 for c in doc_freq.values() if c >= min_df)
    print(f"min_df={min_df:3}: {n} Wörter bleiben")

In [ ]:
total = sum(len(t) for t in sample["tokens"])

for min_df in [2, 5, 10, 20]:
    vocab = {w for w, c in doc_freq.items() if c >= min_df}
    kept = sum(1 for toks in sample["tokens"] for t in toks if t in vocab)
    print(f"min_df={min_df:3}: {kept / total:.1%} aller Wörter bleiben")

In [ ]:
import random
random.seed(42)

for lo, hi in [(2, 5), (5, 10)]:
    wörter = [w for w, c in doc_freq.items() if lo <= c < hi]
    print(f"{lo} bis {hi - 1} Beschwerden ({len(wörter)} Wörter):")
    print(random.sample(wörter, min(40, len(wörter))), "\n")

In [ ]:
MIN_DF = 10   
vocab = {w for w, c in doc_freq.items() if c >= MIN_DF}
sample["tokens_filtered"] = sample["tokens"].apply(lambda toks: [t for t in toks if t in vocab])

sample["n_tokens"] = sample["tokens_filtered"].apply(len)
print(sample["n_tokens"].describe())
print((sample["n_tokens"] < 5).sum(), "Beschwerden mit weniger als 5 Wörtern")

In [ ]:
#Finde herraus was eine sinnvolle Untergrenze ist
for grenze in [1, 3, 5, 8, 10, 15, 20]:
    n = (sample["n_tokens"] < grenze).sum()
    print(f"weniger als {grenze:2} Wörter: {n:4} Beschwerden ({n / len(sample):.1%})")

In [ ]:
kurz = sample[(sample["n_tokens"] >= 3) & (sample["n_tokens"] < 10)].sample(12, random_state=42)
for _, row in kurz.iterrows():
    print(row["n_tokens"], "|", row["text_clean"][:150])
    print("   ", row["tokens_filtered"], "\n")

In [ ]:
MIN_TOKENS = 4   # Einschränkung niedrig angesetzt um möglichst wenig semantischen Verlust zu haben
sample_final = sample[sample["n_tokens"] >= MIN_TOKENS].copy()
print(len(sample), "->", len(sample_final), f"({len(sample) - len(sample_final)} entfernt)")

vergleich = pd.DataFrame({
    "vorher": sample["Product"].value_counts(normalize=True),
    "nachher": sample_final["Product"].value_counts(normalize=True),
})
vergleich["Differenz"] = vergleich["nachher"] - vergleich["vorher"]
print((vergleich * 100).round(1).sort_values("vorher", ascending=False))

In [ ]:
print(sample_final["n_tokens"].describe())
print(sample_final["tokens_filtered"].iloc[0])

In [ ]:
spalten = ["Complaint ID", "Product", "Issue", "text_clean", "tokens_filtered", "n_tokens"]
sample_final[spalten].to_pickle("data/tokens_final.pkl")

check = pd.read_pickle("data/tokens_final.pkl")
print(check.shape)
print(type(check["tokens_filtered"].iloc[0]))